<a href="https://colab.research.google.com/github/sankum14/Tamil-nadu-election-2026-Result-analysis/blob/main/TamilNaduResults.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q plotly==5.24.1 pandas numpy

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_dark"
print("✅ Libraries loaded")

✅ Libraries loaded


In [ ]:
from google.colab import files
import io

uploaded = files.upload()  # pick eci_results_tamilnadu_2026.csv
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"✅ Loaded: {df.shape[0]} rows × {df.shape[1]} cols")
df.head()

Saving eci_results_tamilnadu_2026.csv to eci_results_tamilnadu_2026 (1).csv
✅ Loaded: 4257 rows × 11 cols


,Code,Constituency,Candidate,Party,EVM Votes,Postal Votes,Total Votes,% Votes,Round,Last Updated Time,Last Updated Date
0,S221,GUMMIDIPOONDI - 1,T.J.GOVINDARAJAN,Dravida Munnetra Kazhagam,61922,570,62492,26.88,26/26,11:27 PM,04/05/2026
1,S221,GUMMIDIPOONDI - 1,SUDHAKAR.V,All India Anna Dravida Munnetra Kazhagam,65894,481,66375,28.55,26/26,11:27 PM,04/05/2026
2,S221,GUMMIDIPOONDI - 1,M.JAGADEESAN,Bahujan Samaj Party,695,2,697,0.30,26/26,11:27 PM,04/05/2026
3,S221,GUMMIDIPOONDI - 1,"R.SRIDAR. B.L.,",Naam Tamilar Katchi,4707,49,4756,2.05,26/26,11:27 PM,04/05/2026
4,S221,GUMMIDIPOONDI - 1,SDK SANKAR,Aanaithinthiya Jananayaka Pathukappu Kazhagam,593,6,599,0.26,26/26,11:27 PM,04/05/2026


In [ ]:
df['Party'] = df['Party'].str.strip()
df['Candidate'] = df['Candidate'].str.strip()
df['Constituency'] = df['Constituency'].str.strip()

df['Const_Num'] = df['Constituency'].str.extract(r'-\s*(\d+)$').astype(float)
df['Const_Name'] = df['Constituency'].str.replace(r'\s*-\s*\d+$', '', regex=True).str.strip()

nota_df = df[df['Party'] == 'None of the Above'].copy()
contestants = df[df['Party'] != 'None of the Above'].copy()

# Winners
winners = contestants.loc[contestants.groupby('Code')['Total Votes'].idxmax()].copy()
winners = winners.sort_values('Code').reset_index(drop=True)

# Runners-up
sorted_c = contestants.sort_values(['Code', 'Total Votes'], ascending=[True, False])
runners = sorted_c.groupby('Code').nth(1).reset_index(drop=True)

# Margins
margin_df = winners[['Code','Const_Name','Constituency','Candidate','Party','Total Votes','% Votes']].rename(
    columns={'Candidate':'Winner','Party':'Winner_Party','Total Votes':'Winner_Votes','% Votes':'Winner_Pct'}
)
margin_df = margin_df.merge(
    runners[['Code','Candidate','Party','Total Votes','% Votes']].rename(
        columns={'Candidate':'Runner_Up','Party':'Runner_Party','Total Votes':'Runner_Votes','% Votes':'Runner_Pct'}
    ),
    on='Code'
)
margin_df['Margin'] = margin_df['Winner_Votes'] - margin_df['Runner_Votes']
margin_df['Margin_Pct'] = margin_df['Winner_Pct'] - margin_df['Runner_Pct']

print(f"✅ Winners: {len(winners)} | Tightest: {margin_df['Margin'].min():,} | Biggest: {margin_df['Margin'].max():,}")

✅ Winners: 234 | Tightest: 30 | Biggest: 98,110


In [ ]:
df['Party'] = df['Party'].str.strip()
df['Candidate'] = df['Candidate'].str.strip()
df['Constituency'] = df['Constituency'].str.strip()

df['Const_Num'] = df['Constituency'].str.extract(r'-\s*(\d+)$').astype(float)
df['Const_Name'] = df['Constituency'].str.replace(r'\s*-\s*\d+$', '', regex=True).str.strip()

nota_df = df[df['Party'] == 'None of the Above'].copy()
contestants = df[df['Party'] != 'None of the Above'].copy()

# Winners
winners = contestants.loc[contestants.groupby('Code')['Total Votes'].idxmax()].copy()
winners = winners.sort_values('Code').reset_index(drop=True)

# Runners-up
sorted_c = contestants.sort_values(['Code', 'Total Votes'], ascending=[True, False])
runners = sorted_c.groupby('Code').nth(1).reset_index(drop=True)

# Margins
margin_df = winners[['Code','Const_Name','Constituency','Candidate','Party','Total Votes','% Votes']].rename(
    columns={'Candidate':'Winner','Party':'Winner_Party','Total Votes':'Winner_Votes','% Votes':'Winner_Pct'}
)
margin_df = margin_df.merge(
    runners[['Code','Candidate','Party','Total Votes','% Votes']].rename(
        columns={'Candidate':'Runner_Up','Party':'Runner_Party','Total Votes':'Runner_Votes','% Votes':'Runner_Pct'}
    ),
    on='Code'
)
margin_df['Margin'] = margin_df['Winner_Votes'] - margin_df['Runner_Votes']
margin_df['Margin_Pct'] = margin_df['Winner_Pct'] - margin_df['Runner_Pct']

print(f"✅ Winners: {len(winners)} | Tightest: {margin_df['Margin'].min():,} | Biggest: {margin_df['Margin'].max():,}")

✅ Winners: 234 | Tightest: 30 | Biggest: 98,110


In [ ]:
PARTY_COLORS = {
    'Tamilaga Vettri Kazhagam': '#E50000',
    'Dravida Munnetra Kazhagam': '#DA251C',
    'All India Anna Dravida Munnetra Kazhagam': '#138808',
    'Indian National Congress': '#19AAED',
    'Bharatiya Janata Party': '#FF9933',
    'Pattali Makkal Katchi': '#FFD700',
    'Naam Tamilar Katchi': '#C71585',
    'Communist Party of India (Marxist)': '#B71C1C',
    'Communist Party of India': '#D32F2F',
    'Viduthalai Chiruthaigal Katchi': '#1A237E',
    'Indian Union Muslim League': '#006A4E',
    'Desiya Murpokku Dravida Kazhagam': '#FF6F00',
    'Amma Makkal Munnettra Kazagam': '#7B1FA2',
    'Independent': '#9E9E9E',
    'None of the Above': '#424242',
}
PARTY_SHORT = {
    'Tamilaga Vettri Kazhagam': 'TVK',
    'Dravida Munnetra Kazhagam': 'DMK',
    'All India Anna Dravida Munnetra Kazhagam': 'AIADMK',
    'Indian National Congress': 'INC',
    'Bharatiya Janata Party': 'BJP',
    'Pattali Makkal Katchi': 'PMK',
    'Naam Tamilar Katchi': 'NTK',
    'Communist Party of India (Marxist)': 'CPI(M)',
    'Communist Party of India': 'CPI',
    'Viduthalai Chiruthaigal Katchi': 'VCK',
    'Indian Union Muslim League': 'IUML',
    'Desiya Murpokku Dravida Kazhagam': 'DMDK',
    'Amma Makkal Munnettra Kazagam': 'AMMK',
    'Independent': 'IND',
    'None of the Above': 'NOTA',
}
def party_color(p): return PARTY_COLORS.get(p, '#607D8B')
def party_short(p): return PARTY_SHORT.get(p, p[:10])
print("✅ Colors ready")

✅ Colors ready


In [ ]:
seat_tally = winners['Party'].value_counts()
total_seats = 234
majority_mark = total_seats // 2 + 1
total_votes_cast = df['Total Votes'].sum()
turnout_candidates = len(contestants)
nota_total = nota_df['Total Votes'].sum()
nota_pct = 100 * nota_total / total_votes_cast

leader_party = seat_tally.index[0]
leader_seats = seat_tally.iloc[0]
runner_party = seat_tally.index[1]
runner_seats = seat_tally.iloc[1]

card_html = f'''
<style>
.news-grid {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap:14px; padding:14px; font-family: -apple-system, "Segoe UI", Roboto, sans-serif; }}
.news-card {{ padding:18px 20px; border-radius:14px; color:#fff; box-shadow:0 6px 20px rgba(0,0,0,.35); position:relative; overflow:hidden; }}
.news-card::before {{ content:""; position:absolute; top:0; left:0; right:0; height:3px; background:rgba(255,255,255,.5); }}
.news-card .label {{ font-size:11px; letter-spacing:1.5px; text-transform:uppercase; opacity:.85; margin-bottom:8px; }}
.news-card .value {{ font-size:32px; font-weight:800; line-height:1; margin-bottom:6px; }}
.news-card .sub {{ font-size:13px; opacity:.85; }}
.c1 {{ background: linear-gradient(135deg, #E50000, #8B0000); }}
.c2 {{ background: linear-gradient(135deg, #DA251C, #6B1010); }}
.c3 {{ background: linear-gradient(135deg, #138808, #0a5805); }}
.c4 {{ background: linear-gradient(135deg, #1976D2, #0D47A1); }}
.c5 {{ background: linear-gradient(135deg, #FF6F00, #BF360C); }}
.c6 {{ background: linear-gradient(135deg, #424242, #1c1c1c); }}
</style>
<div class="news-grid">
  <div class="news-card c1"><div class="label">🏆 Winning Party</div><div class="value">{party_short(leader_party)}</div><div class="sub">{leader_seats} of {total_seats} seats · {"MAJORITY" if leader_seats >= majority_mark else f"{majority_mark - leader_seats} short"}</div></div>
  <div class="news-card c2"><div class="label">🥈 Runner-up</div><div class="value">{party_short(runner_party)}</div><div class="sub">{runner_seats} seats · {leader_seats - runner_seats} behind</div></div>
  <div class="news-card c3"><div class="label">📊 Magic Number</div><div class="value">{majority_mark}</div><div class="sub">Seats needed for majority</div></div>
  <div class="news-card c4"><div class="label">🗳️ Total Votes</div><div class="value">{total_votes_cast/1e7:.2f} Cr</div><div class="sub">Across {total_seats} seats</div></div>
  <div class="news-card c5"><div class="label">👥 Candidates</div><div class="value">{turnout_candidates:,}</div><div class="sub">Avg {turnout_candidates/total_seats:.0f} per seat</div></div>
  <div class="news-card c6"><div class="label">🚫 NOTA</div><div class="value">{nota_total/1e5:.1f}L</div><div class="sub">{nota_pct:.2f}% of votes</div></div>
</div>
'''
display(HTML(card_html))

In [ ]:
seat_tally_full = winners['Party'].value_counts()
ordered_parties = seat_tally_full.index.tolist()

n_seats = 234
n_rows = 8
radii = np.linspace(1.0, 1.7, n_rows)
weights = radii / radii.sum()
counts = np.floor(weights * n_seats).astype(int)
while counts.sum() < n_seats:
    counts[np.argmax(weights * n_seats - counts)] += 1
while counts.sum() > n_seats:
    counts[np.argmin(weights * n_seats - counts)] -= 1

points = []
for r, c in zip(radii, counts):
    if c == 0: continue
    angles = np.linspace(np.pi, 0, c)
    for a in angles:
        points.append((r * np.cos(a), r * np.sin(a), a))
points.sort(key=lambda p: -p[2])

party_per_seat = []
for p in ordered_parties:
    party_per_seat.extend([p] * seat_tally_full[p])
party_per_seat = party_per_seat[:n_seats]

xs = [p[0] for p in points]
ys = [p[1] for p in points]

fig = go.Figure()
for party in ordered_parties:
    idx = [i for i, p in enumerate(party_per_seat) if p == party]
    if not idx: continue
    fig.add_trace(go.Scatter(
        x=[xs[i] for i in idx], y=[ys[i] for i in idx], mode='markers',
        marker=dict(size=18, color=party_color(party), line=dict(color='white', width=1)),
        name=f"{party_short(party)} ({seat_tally_full[party]})",
        hovertemplate=f"<b>{party}</b><br>Seats: {seat_tally_full[party]}<extra></extra>"
    ))

fig.add_annotation(x=0, y=-0.15, text=f"<b>Majority Mark: {majority_mark}</b>",
    showarrow=False, font=dict(size=14, color='#FFD700'))
fig.add_annotation(x=0, y=1.85,
    text=f"<b style='font-size:24px'>Tamil Nadu Assembly 2026</b><br><span style='font-size:14px'>{party_short(leader_party)} wins {leader_seats} of {total_seats}</span>",
    showarrow=False, font=dict(color='white'))

fig.update_layout(
    showlegend=True,
    xaxis=dict(visible=False, range=[-2, 2]),
    yaxis=dict(visible=False, range=[-0.4, 2.2], scaleanchor='x', scaleratio=1),
    height=600, plot_bgcolor='#0a0a0a', paper_bgcolor='#0a0a0a',
    legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center', font=dict(size=11)),
    margin=dict(l=20, r=20, t=20, b=20)
)
fig.show()

In [ ]:
party_votes = contestants.groupby('Party')['Total Votes'].sum().sort_values(ascending=False)
party_vote_pct = (party_votes / party_votes.sum() * 100)
party_seats = winners['Party'].value_counts()
party_seat_pct = (party_seats / party_seats.sum() * 100)

top_parties = party_vote_pct.head(8).index.tolist()
for p in party_seats.head(8).index:
    if p not in top_parties:
        top_parties.append(p)

vshare = [party_vote_pct.get(p, 0) for p in top_parties]
sshare = [party_seat_pct.get(p, 0) for p in top_parties]
colors_list = [party_color(p) for p in top_parties]
labels_short = [party_short(p) for p in top_parties]

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=('<b>VOTE SHARE</b>', '<b>SEAT SHARE</b>'))

fig.add_trace(go.Pie(labels=labels_short, values=vshare, hole=.55,
    marker=dict(colors=colors_list, line=dict(color='#0a0a0a', width=2)),
    textinfo='label+percent', textfont=dict(size=12, color='white'),
    hovertemplate='<b>%{label}</b><br>%{value:.2f}%<extra></extra>', sort=False), 1, 1)
fig.add_trace(go.Pie(labels=labels_short, values=sshare, hole=.55,
    marker=dict(colors=colors_list, line=dict(color='#0a0a0a', width=2)),
    textinfo='label+percent', textfont=dict(size=12, color='white'),
    hovertemplate='<b>%{label}</b><br>%{value:.2f}%<extra></extra>', sort=False), 1, 2)

fig.update_layout(
    title=dict(text='<b>Vote Share vs Seat Share — The FPTP Effect</b>', x=0.5, font=dict(size=20)),
    height=550, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a', showlegend=False,
    annotations=[dict(text='VOTES', x=0.18, y=0.5, font=dict(size=16, color='white'), showarrow=False),
                 dict(text='SEATS', x=0.82, y=0.5, font=dict(size=16, color='white'), showarrow=False)]
)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Vote %', x=labels_short, y=vshare, marker_color='#888',
    text=[f"{v:.1f}%" for v in vshare], textposition='outside'))
fig2.add_trace(go.Bar(name='Seat %', x=labels_short, y=sshare, marker_color=colors_list,
    text=[f"{v:.1f}%" for v in sshare], textposition='outside'))
fig2.update_layout(barmode='group', height=450,
    title=dict(text='<b>Vote % vs Seat % — Side by Side</b>', x=0.5),
    paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    yaxis=dict(title='%', gridcolor='#333'),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig2.show()

In [ ]:
max_margin = margin_df['Margin'].max()
raw_bins = [0, 1000, 5000, 10000, 25000, 50000, 100000, 200000]
bins = [b for b in raw_bins if b < max_margin] + [max_margin + 1]
all_labels = ['<1K','1K-5K','5K-10K','10K-25K','25K-50K','50K-100K','100K-200K','200K+']
labels = all_labels[:len(bins)-1]
margin_df['Margin_Band'] = pd.cut(margin_df['Margin'], bins=bins, labels=labels, include_lowest=True)
band_counts = margin_df['Margin_Band'].value_counts().reindex(labels)

all_band_colors = ['#FF1744','#FF5722','#FF9800','#FFC107','#CDDC39','#8BC34A','#4CAF50','#2E7D32']
band_colors = all_band_colors[:len(labels)]

fig = go.Figure(go.Bar(x=labels, y=band_counts.values,
    marker=dict(color=band_colors, line=dict(color='white', width=1)),
    text=band_counts.values, textposition='outside', textfont=dict(size=14, color='white'),
    hovertemplate='<b>Margin %{x}</b><br>%{y} constituencies<extra></extra>'))
fig.update_layout(
    title=dict(text='<b>Distribution of Victory Margins</b>', x=0.5, font=dict(size=18)),
    xaxis_title='Margin Band (votes)', yaxis_title='Constituencies',
    height=480, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    yaxis=dict(gridcolor='#333'))
fig.show()

print(f"Median margin: {margin_df['Margin'].median():,.0f}")
print(f"Closest:       {margin_df['Margin'].min():,.0f}")
print(f"Biggest:       {margin_df['Margin'].max():,.0f}")
print(f"Won by < 5K:   {(margin_df['Margin'] < 5000).sum()} seats")
print(f"Won by > 50K:  {(margin_df['Margin'] > 50000).sum()} seats")

Median margin: 11,416
Closest:       30
Biggest:       98,110
Won by < 5K:   61 seats
Won by > 50K:  15 seats


In [ ]:
top20 = margin_df.nlargest(20, 'Margin').copy()
top20['Label'] = top20['Winner'].str.title() + '<br><span style="font-size:10px">' + top20['Const_Name'] + ' · ' + top20['Winner_Party'].map(PARTY_SHORT).fillna(top20['Winner_Party']) + '</span>'
top20 = top20.iloc[::-1]

fig = go.Figure(go.Bar(
    y=top20['Label'], x=top20['Margin'], orientation='h',
    marker=dict(color=[party_color(p) for p in top20['Winner_Party']], line=dict(color='white', width=1)),
    text=[f"{m:,}" for m in top20['Margin']],
    textposition='outside', textfont=dict(color='white', size=11),
    hovertemplate='<b>%{y}</b><br>Margin: %{x:,}<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>🏆 Top 20 Biggest Victory Margins</b>', x=0.5, font=dict(size=18)),
    xaxis_title='Margin (votes)', height=750,
    paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    xaxis=dict(gridcolor='#333'), margin=dict(l=180))
fig.show()

In [ ]:
closest = margin_df.nsmallest(15, 'Margin').copy().iloc[::-1]

fig = go.Figure()
for i, row in closest.iterrows():
    fig.add_trace(go.Bar(
        y=[row['Const_Name'][:24]], x=[row['Winner_Votes']], orientation='h',
        marker=dict(color=party_color(row['Winner_Party'])), showlegend=False,
        text=f"<b>{row['Winner'].split()[0].title()}</b> · {row['Winner_Votes']:,}",
        textposition='inside', textfont=dict(color='white', size=10),
        hovertemplate=f"<b>{row['Winner']}</b> ({party_short(row['Winner_Party'])})<br>Won by: {row['Margin']:,}<extra></extra>"
    ))
    fig.add_trace(go.Bar(
        y=[row['Const_Name'][:24]], x=[row['Runner_Votes']], orientation='h',
        marker=dict(color=party_color(row['Runner_Party']), opacity=0.7), showlegend=False,
        text=f"{row['Runner_Up'].split()[0].title()} · {row['Runner_Votes']:,}",
        textposition='inside', textfont=dict(color='white', size=10),
        hovertemplate=f"<b>{row['Runner_Up']}</b> ({party_short(row['Runner_Party'])})<extra></extra>"
    ))

fig.update_layout(barmode='stack',
    title=dict(text='<b>⚔️ Top 15 Closest Contests</b>', x=0.5, font=dict(size=18)),
    height=700, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    xaxis=dict(title='Votes', gridcolor='#333'), margin=dict(l=180))

for i, (_, row) in enumerate(closest.iterrows()):
    fig.add_annotation(x=row['Winner_Votes']+row['Runner_Votes']+1500, y=i,
        text=f"<b>Δ {row['Margin']:,}</b>", showarrow=False, font=dict(color='#FF1744', size=12))
fig.show()

In [ ]:
major_parties = seat_tally[seat_tally >= 1].index.tolist()

fig = go.Figure()
buttons = []
for i, party in enumerate(major_parties):
    party_wins = margin_df[margin_df['Winner_Party'] == party].sort_values('Margin', ascending=True)
    if len(party_wins) == 0: continue
    fig.add_trace(go.Bar(
        y=party_wins['Const_Name'], x=party_wins['Margin'], orientation='h',
        marker=dict(color=party_color(party), line=dict(color='white', width=0.5)),
        text=[f"{m:,}" for m in party_wins['Margin']],
        textposition='outside', textfont=dict(color='white', size=10),
        name=party_short(party), visible=(i==0),
        hovertemplate='<b>%{y}</b><br>Margin: %{x:,}<extra></extra>'
    ))

for i, party in enumerate(major_parties):
    visibility = [j == i for j in range(len(major_parties))]
    buttons.append(dict(label=f"{party_short(party)} ({seat_tally[party]})",
        method='update',
        args=[{'visible': visibility},
              {'title': f'<b>{party} — {seat_tally[party]} Wins</b>'}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction='down', showactive=True,
        x=0.02, y=1.12, xanchor='left', yanchor='top',
        bgcolor='#1a1a1a', bordercolor='#444', font=dict(color='white'))],
    title=dict(text=f'<b>{major_parties[0]} — {seat_tally[major_parties[0]]} Wins</b>', x=0.5),
    height=max(450, 22*seat_tally.iloc[0]),
    paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    xaxis=dict(title='Margin', gridcolor='#333'), margin=dict(l=200, t=120))
fig.show()

In [ ]:
contested = contestants.groupby('Party')['Code'].nunique()
won = winners['Party'].value_counts()
strike = pd.DataFrame({'Contested': contested, 'Won': won}).fillna(0)
strike['Strike Rate %'] = (strike['Won'] / strike['Contested'] * 100).round(2)
strike = strike[strike['Contested'] >= 10].sort_values('Strike Rate %', ascending=False).head(15)

fig = go.Figure(go.Bar(
    x=[party_short(p) for p in strike.index], y=strike['Strike Rate %'],
    marker=dict(color=[party_color(p) for p in strike.index], line=dict(color='white', width=1)),
    text=[f"{v:.1f}%<br><span style='font-size:10px'>{int(strike.loc[p,'Won'])}/{int(strike.loc[p,'Contested'])}</span>" for p,v in strike['Strike Rate %'].items()],
    textposition='outside', textfont=dict(color='white', size=11),
    hovertemplate='<b>%{x}</b><br>%{y:.1f}%<extra></extra>'
))
fig.update_layout(
    title=dict(text='<b>🎯 Strike Rate (min 10 contested)</b>', x=0.5, font=dict(size=18)),
    yaxis_title='Strike Rate %', height=500,
    paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    yaxis=dict(gridcolor='#333', range=[0, max(105, strike['Strike Rate %'].max()*1.15)]))
fig.show()
display(strike.style.background_gradient(subset=['Strike Rate %'], cmap='RdYlGn').format({'Strike Rate %':'{:.2f}%','Contested':'{:.0f}','Won':'{:.0f}'}))

,Contested,Won,Strike Rate %
Party,,,
Tamilaga Vettri Kazhagam,233,107,45.92%
Dravida Munnetra Kazhagam,176,60,34.09%
All India Anna Dravida Munnetra Kazhagam,172,47,27.33%
Pattali Makkal Katchi,18,4,22.22%
Indian National Congress,28,5,17.86%
Desiya Murpokku Dravida Kazhagam,10,1,10.00%
Amma Makkal Munnettra Kazagam,11,1,9.09%
Bharatiya Janata Party,33,1,3.03%
Anti Corruption Dynamic Party,14,0,0.00%


In [ ]:
labels_all, parents_all, values_all, colors_all = [], [], [], []
tm_df = winners.copy()
tm_df['Party_Short'] = tm_df['Party'].map(PARTY_SHORT).fillna(tm_df['Party'])

for p_short in tm_df['Party_Short'].unique():
    subset = tm_df[tm_df['Party_Short'] == p_short]
    p_full = subset['Party'].iloc[0]
    n_seats = len(subset)
    labels_all.append(f"{p_short} ({n_seats})")
    parents_all.append("")
    values_all.append(subset['Total Votes'].sum())
    colors_all.append(party_color(p_full))
    for _, row in subset.iterrows():
        labels_all.append(row['Const_Name'].title())
        parents_all.append(f"{p_short} ({n_seats})")
        values_all.append(row['Total Votes'])
        colors_all.append(party_color(row['Party']))

fig = go.Figure(go.Treemap(
    labels=labels_all, parents=parents_all, values=values_all,
    marker=dict(colors=colors_all, line=dict(color='#0a0a0a', width=1)),
    textinfo='label+value', textfont=dict(size=10, color='white'),
    hovertemplate='<b>%{label}</b><br>Votes: %{value:,}<extra></extra>',
    branchvalues='total'))
fig.update_layout(
    title=dict(text='<b>🌊 Wave Map — 234 Constituencies by Winner Votes</b>', x=0.5, font=dict(size=18)),
    height=700, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    margin=dict(l=10, r=10, t=60, b=10))
fig.show()

In [ ]:
big3 = ['Tamilaga Vettri Kazhagam','Dravida Munnetra Kazhagam','All India Anna Dravida Munnetra Kazhagam']
big3_short = ['TVK','DMK','AIADMK']

positions = {p: {'1st':0,'2nd':0,'3rd':0,'Below 3rd':0} for p in big3}
for code, group in contestants.groupby('Code'):
    g = group.sort_values('Total Votes', ascending=False).reset_index(drop=True)
    for p in big3:
        match = g[g['Party'] == p]
        if len(match) == 0: continue
        rank = match.index[0] + 1
        if rank == 1: positions[p]['1st'] += 1
        elif rank == 2: positions[p]['2nd'] += 1
        elif rank == 3: positions[p]['3rd'] += 1
        else: positions[p]['Below 3rd'] += 1

fig = go.Figure()
for cat, color in zip(['1st','2nd','3rd','Below 3rd'], ['#FFD700','#C0C0C0','#CD7F32','#555']):
    fig.add_trace(go.Bar(name=cat, x=big3_short,
        y=[positions[p][cat] for p in big3], marker_color=color,
        text=[positions[p][cat] for p in big3], textposition='inside',
        textfont=dict(color='black', size=14)))
fig.update_layout(barmode='stack',
    title=dict(text='<b>👑 TVK vs DMK vs AIADMK — Where Each Finished</b>', x=0.5, font=dict(size=18)),
    height=500, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    yaxis=dict(title='Constituencies', gridcolor='#333'),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig.show()

big3_totals = contestants[contestants['Party'].isin(big3)].groupby('Party')['Total Votes'].sum()
fig2 = go.Figure(go.Bar(
    x=[party_short(p) for p in big3_totals.index], y=big3_totals.values,
    marker_color=[party_color(p) for p in big3_totals.index],
    text=[f"{v/1e7:.2f} Cr<br>({v/contestants['Total Votes'].sum()*100:.1f}%)" for v in big3_totals.values],
    textposition='outside', textfont=dict(color='white', size=14)))
fig2.update_layout(
    title=dict(text='<b>Total Votes Polled — Big 3</b>', x=0.5, font=dict(size=18)),
    height=420, paper_bgcolor='#0a0a0a', plot_bgcolor='#0a0a0a',
    yaxis=dict(title='Votes', gridcolor='#333'))
fig2.show()

In [1]:
git remote add origin https://github.com/sankum14/Tamil-nadu-election-2026-Result-analysis.git
git branch -M main
git push -u origin main

SyntaxError: invalid syntax (3159483756.py, line 1)